# RAG — Creation stage: Features + test cases

**Goal:** Explain **how** the system builds a **feature** list and then **test cases** per feature from an SRS-style document (e.g. JDECo), including **counts**, **where results live**, and **where prompts are defined** in code.

**Pipeline (summary):**
1. Load file → text → chunk → **in-memory FAISS** (same idea as earlier RAG lab stages).
2. **Feature extraction:** LLM over document context (one pass for short docs, or **segments + merge** for long ones).
3. **Test generation:** For each feature, **retrieve** nearest chunks (multi-query merge + light lexical reranking), then LLM emits JSON `testCases` with **evidence** tied to chunk ids.

**Requirement:** `OPENAI_API_KEY` in `rag_lab/.env` (or the environment), because steps 2 and 3 call the model.

---

### Where things live in the repo

| What | File |
|------|------|
| End-to-end runner | `src/qbrain_rag/application/document_pipeline.py` → `run_document_pipeline` |
| Feature + test prompt templates | `src/qbrain_rag/application/prompts_srs.py` |
| Feature extraction (single vs segment+merge) | `src/qbrain_rag/application/feature_extraction.py` |
| Context retrieval + test generation + quality filters | `src/qbrain_rag/application/test_case_generation.py` |
| CLI (prints JSON) | `scripts/run_document_pipeline.py` |
| Example saved outputs (if present) | `results/pipeline_runs/` |


### Flow diagram

```mermaid
flowchart TD
  A[PDF / text] --> B[load_document]
  B --> C[chunk_text]
  C --> D[build_faiss_store]
  D --> E{Chunk count}
  E -->|≤ 5 by default| F[LLM: feature extraction — single pass]
  E -->|more| G[LLM: segments then merge features]
  F --> H[features list]
  G --> H
  H --> I[Per feature: multi similarity_search]
  I --> J[LLM: testCases + evidence]
```

Constants such as `chunks_per_group` and `max_context_chars` are set in `feature_extraction.py`. Per-test context breadth is controlled by `n_test_context_chunks` on `run_document_pipeline`.


In [ ]:
%pip install -q langchain-text-splitters langchain-core langchain-community langchain-openai faiss-cpu pymupdf openai python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Prompt excerpts (live code)

The snippets below are read from the same modules so this notebook stays in sync when prompts change.


In [ ]:
import inspect
import runpy
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_roots = []
if _cwd.name == "notebooks":
    _roots.append(_cwd.parent)
_roots.extend(
    [
        _cwd,
        _cwd / "QBrain" / "rag_lab",
        _cwd / "rag_lab",
        _cwd.parent,
    ]
)
_roots.extend(_cwd.parents)
_jb = None
for _r in _roots:
    _p = Path(_r).resolve() / "jupyter_bootstrap.py"
    if _p.is_file() and (_p.parent / "src" / "qbrain_rag").is_dir():
        _jb = _p
        break
if _jb is None:
    raise RuntimeError(
        "Could not find rag_lab/jupyter_bootstrap.py. "
        f"Open QBrain/rag_lab or QBrain/rag_lab/notebooks as the working folder. cwd={_cwd!s}"
    )
_ns = runpy.run_path(str(_jb))
RAG_LAB = _ns["RAG_LAB"]

import qbrain_rag.application.prompts_srs as prompts_srs
import qbrain_rag.application.feature_extraction as feature_extraction
import qbrain_rag.application.test_case_generation as test_case_generation
import qbrain_rag.application.document_pipeline as document_pipeline

def loc(obj):
    return Path(inspect.getfile(obj)).resolve().relative_to(RAG_LAB)

print("Module paths (under rag_lab/):")
print("  prompts_srs          ", loc(prompts_srs))
print("  feature_extraction   ", loc(feature_extraction))
print("  test_case_generation ", loc(test_case_generation))
print("  document_pipeline    ", loc(document_pipeline))
print()

print("--- create_adaptive_prompt() (feature rules) ---")
print(prompts_srs.create_adaptive_prompt()[:2200])
print("\n... [truncated] ...\n")

print("--- FEATURE_EXTRACTION_USER_TEMPLATE (full-document message shape) ---")
print(prompts_srs.FEATURE_EXTRACTION_USER_TEMPLATE[:900])
print("...\n")

sample_user = prompts_srs.build_test_case_user_prompt(
    feature_description="(feature description from extraction step)",
    context="(retrieved chunk text only)",
    feature_type="FUNCTIONAL",
    matched_sections=["example phrase from document"],
)
print("--- build_test_case_user_prompt (sample; real context comes from retrieval) ---")
print(sample_user[:1600])
print("...\n")


Module paths (under rag_lab/):
  prompts_srs           src\qbrain_rag\application\prompts_srs.py
  feature_extraction    src\qbrain_rag\application\feature_extraction.py
  test_case_generation  src\qbrain_rag\application\test_case_generation.py
  document_pipeline     src\qbrain_rag\application\document_pipeline.py

--- create_adaptive_prompt() (feature rules) ---

**Context:** Chunks appear in reading order (`#1`, `#2`, …). If the statistics line says truncated, only that prefix exists — do not invent the rest.

**Essential**
- Extract only **testable obligations** grounded in the text (behaviour, data, workflows, constraints). Skip pure marketing with no checkable claim.
- **Merge** paraphrases of the same obligation into one feature. Split only for clearly separate obligations (different actors, outcomes, or stable labels).
- **Do not invent** requirements, products, or integrations absent from the context.

**Preferred**
- One feature per explicit label (req ID, ticket key, repeate

### Run the pipeline on JDECo (or default SRS path)

- **`max_features`:** Cap how many features get test cases (faster demos). Set to `None` for all features.
- **`skip_test_cases=True`:** Features only, no test-generation LLM calls.
- Progress logs go to **stderr** (Jupyter shows them in the execution stream).


In [ ]:
from collections import Counter
import json
import runpy
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_roots = []
if _cwd.name == "notebooks":
    _roots.append(_cwd.parent)
_roots.extend(
    [
        _cwd,
        _cwd / "QBrain" / "rag_lab",
        _cwd / "rag_lab",
        _cwd.parent,
    ]
)
_roots.extend(_cwd.parents)
_jb = None
for _r in _roots:
    _p = Path(_r).resolve() / "jupyter_bootstrap.py"
    if _p.is_file() and (_p.parent / "src" / "qbrain_rag").is_dir():
        _jb = _p
        break
if _jb is None:
    raise RuntimeError(
        "Could not find rag_lab/jupyter_bootstrap.py. "
        f"Open QBrain/rag_lab or QBrain/rag_lab/notebooks as the working folder. cwd={_cwd!s}"
    )
_ns = runpy.run_path(str(_jb))
RAG_LAB = _ns["RAG_LAB"]

from qbrain_rag.application.document_pipeline import run_document_pipeline
from qbrain_rag.config.notebook_defaults import resolve_default_srs_path

srs_path = resolve_default_srs_path(RAG_LAB)
print("SRS:", srs_path.resolve())

# Adjust: None = all features (slower, more API cost)
MAX_FEATURES_FOR_DEMO = 5
N_TEST_CONTEXT_CHUNKS = 5

result = run_document_pipeline(
    srs_path,
    n_test_context_chunks=N_TEST_CONTEXT_CHUNKS,
    max_features=MAX_FEATURES_FOR_DEMO,
    skip_test_cases=False,
    verbose=True,
)


[doc] Loading: D:\Qbrainpython\QBrain\rag_lab\data\srs\JDECo_SRS.docx[1].pdf
[doc] Text length: 45,175 chars
[doc] Chunks: 30 — building FAISS...


SRS: D:\Qbrainpython\QBrain\rag_lab\data\srs\JDECo_SRS.docx[1].pdf


[doc] Extracting features (LLM)...
[doc] Features: 65 total, using 5 for tests.
[doc] Tests for feature 1/5: 'Service Request Management'...
[doc]   → 4 test case(s)
[doc] Tests for feature 2/5: 'Service Status Tracking'...
[doc]   → 5 test case(s)
[doc] Tests for feature 3/5: 'SMS Notifications for Service Status'...
[doc]   → 5 test case(s)
[doc] Tests for feature 4/5: 'User Authentication'...
[doc]   → 3 test case(s)
[doc] Tests for feature 5/5: 'Integration with Other Systems'...
[doc]   → 5 test case(s)
[doc] Done.


### Counts summary and JSON layout

- **`result["features"]`:** Each item has LLM feature fields, `testCases`, and usually **`evidence`** (retrieval queries, `retrieved_chunk_ids`, `matched_sections`).
- **`result["metadata"]`:** Describes extraction mode (`single_pass` or `segment_then_merge`), `segment_count`, `total_chunks`, etc.
- **Disk export:** The next cell writes JSON under `results/pipeline_runs/` (tracking in git depends on project rules).


In [ ]:
features = result["features"]
meta = result["metadata"]
chunk_count = result["chunk_count"]
text_len = result["text_length"]

n_features = len(features)
n_tests = sum(len(f.get("testCases") or []) for f in features)
by_type = Counter(str(f.get("featureType") or "(missing)") for f in features)

print("=== Creation summary ===")
print(f"Source: {result['source_file']}")
print(f"Text length: {text_len:,} characters")
print(f"Chunk count (FAISS index): {chunk_count}")
print(f"Feature extraction mode: {meta.get('extraction_mode')}  |  segments: {meta.get('segment_count')}  |  truncated: {meta.get('context_truncated')}")
print(f"Features in this run: {n_features}  (after max_features cap if any)")
print(f"Total test cases (after quality filter): {n_tests}")
print("Feature type counts:", dict(by_type))
print()

print("=== Per feature: name, test count, evidence chunk ids ===")
for i, f in enumerate(features, 1):
    name = (f.get("name") or "?")[:70]
    tcs = f.get("testCases") or []
    ev = f.get("evidence") or {}
    cids = ev.get("retrieved_chunk_ids", [])
    print(f"{i:2}. {name!r}  |  tests: {len(tcs):2}  |  retrieved_chunk_ids: {cids}")

if n_tests and features[0].get("testCases"):
    tc0 = features[0]["testCases"][0]
    print("\n=== First test case (excerpt) ===")
    print("title:", tc0.get("title"))
    print("per-case evidence:", tc0.get("evidence"))


=== Creation summary ===
Source: JDECo_SRS.docx[1].pdf
Text length: 45,175 characters
Chunk count (FAISS index): 30
Feature extraction mode: segment_then_merge  |  segments: 6  |  truncated: False
Features in this run: 5  (after max_features cap if any)
Total test cases (after quality filter): 22
Feature type counts: {'FUNCTIONAL': 4, 'NOTIFICATION': 1}

=== Per feature: name, test count, evidence chunk ids ===
 1. 'Service Request Management'  |  tests:  4  |  retrieved_chunk_ids: [28, 20, 16, 6, 26]
 2. 'Service Status Tracking'  |  tests:  5  |  retrieved_chunk_ids: [16, 20, 8, 10, 21]
 3. 'SMS Notifications for Service Status'  |  tests:  5  |  retrieved_chunk_ids: [7, 16, 8, 6, 15]
 4. 'User Authentication'  |  tests:  3  |  retrieved_chunk_ids: [25, 5, 22, 11, 24]
 5. 'Integration with Other Systems'  |  tests:  5  |  retrieved_chunk_ids: [4, 24, 11, 23, 25]

=== First test case (excerpt) ===
title: Verify service request submission with valid data
per-case evidence: {'retrieved_

In [ ]:
import runpy
from pathlib import Path

_cwd = Path.cwd().resolve()
_roots = []
if _cwd.name == "notebooks":
    _roots.append(_cwd.parent)
_roots.extend(
    [
        _cwd,
        _cwd / "QBrain" / "rag_lab",
        _cwd / "rag_lab",
        _cwd.parent,
    ]
)
_roots.extend(_cwd.parents)
_jb = None
for _r in _roots:
    _p = Path(_r).resolve() / "jupyter_bootstrap.py"
    if _p.is_file() and (_p.parent / "src" / "qbrain_rag").is_dir():
        _jb = _p
        break
if _jb is None:
    raise RuntimeError(
        "Could not find rag_lab/jupyter_bootstrap.py. "
        f"Open QBrain/rag_lab or QBrain/rag_lab/notebooks as the working folder. cwd={_cwd!s}"
    )
_ns = runpy.run_path(str(_jb))
RAG_LAB = _ns["RAG_LAB"]
out_dir = RAG_LAB / "results" / "pipeline_runs"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"creation_{result['source_file']}.json"
# Windows-safe filename: replace forbidden characters
safe_name = "".join(c if c not in '<>:"/\\|?*[]' else "_" for c in out_path.name)
out_path = out_dir / safe_name

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print("Saved:", out_path.resolve())


Saved: D:\Qbrainpython\QBrain\rag_lab\results\pipeline_runs\creation_JDECo_SRS.docx_1_.pdf.json
